### **FIMbench - `query`**

Query benchmark flood inundation maps (FIMs) from the FIMbench database and,
optionally, download the matched assets (GeoTIFF + AOI GeoPackage).

This is the **read side**: it reads the unified catalog core published by
`publish` and returns plain Python dicts, so it needs **no AWS credentials** -
the catalog and assets are served as public, anonymous S3 reads.

### **Connecting to an evaluation framework**

`benchFIMquery` is built to drop into any FIM evaluation pipeline:
- **Plain-dict output** (`status`, `message`, `matches`, `printable`) - serializes
  to JSON and is easy to assert on in tests.
- **AOI-driven matching** - pass your model's predicted raster/boundary and get the
  benchmark records covering that area: the natural ground-truth pair for a
  contingency / CSI / POD-FAR metric.
- **Local paths on demand** - with `download=True`, each match carries the local
  benchmark raster + GeoPackage paths to feed your evaluator.

Based on the tier, area overlap, event dates, or any other preference, a user can
select exactly which benchmark FIMs to pull and feed downstream.

**Works seamlessly with FIMeval**

`fimbench.query` powers benchmark access in
[**FIMeval**](https://github.com/sdmlua/fimeval), SDML's flood-map evaluation
framework. FIMeval uses `benchFIMquery` to discover and download the right
benchmark FIM for an area/event, then evaluates a predicted flood map against it -
query a benchmark, access the assets, and evaluate, with no glue code.

### **Install**

In [ ]:
!uv pip install fimbench

### **All parameters**

Every `benchFIMquery` parameter is **optional** and keyword-only. The cell below
lists them all with a comment; combine the ones you need (later scenarios show
common combinations).

In [ ]:
from fimbench import benchFIMquery

response = benchFIMquery(
    raster_path=None,    # path to your predicted-FIM raster -> AOI for spatial search
    boundary_path=None,  # path to a vector AOI (.gpkg/.shp) -> AOI for spatial search
    huc8=None,           # restrict to a HUC8 basin id, e.g. '03020201'
    event_date=None,     # exact event date, e.g. '2017-08-30' (or '...T16')
    start_date=None,     # inclusive range start, e.g. '2016-04-01'
    end_date=None,       # inclusive range end,   e.g. '2026-01-01'
    file_name=None,      # exact catalog filename(s); str or list[str]
    tier=None,           # 'HWM', 'tier_1', 'Tier 2', 'tier3', ... (flexible)
    area=False,          # True -> add overlap %% and km2 per match (needs an AOI)
    download=False,      # True -> download matched tif + gpkg
    out_dir=None,        # download dir; falls back to the AOI file's folder
)
print(response)          # pretty summary; response is a plain dict

### **Scenario 1 - just query (no download): discover what exists**

In [ ]:
response = benchFIMquery(
    start_date='2016-04-01',  # range start
    end_date='2026-01-01',    # range end
    # huc8='03020201',        # optionally narrow by basin
    # tier='tier4',           # optionally narrow by tier
)
records = [m['record'] for m in response['matches']]
len(records)

### **Scenario 2 - AOI search against your predicted FIM, with overlap stats**

In [ ]:
response = benchFIMquery(
    raster_path='path/to/your_predicted_fim.tif',  # your model output as the AOI
    area=True,                                     # add overlap %% and km2 per match
)
print(response)

### **Scenario 3 - match an AOI for an event and download the assets**

In [ ]:
response = benchFIMquery(
    boundary_path='path/to/aoi.gpkg',  # AOI boundary
    event_date='2017-08-30',           # exact event date
    download=True,                     # fetch matched tif + gpkg
    out_dir='../downloads/',           # download destination (optional)
)
for m in response['matches']:
    print(m['downloads'])  # local tif / gpkg paths to feed your evaluator

### **Scenario 4 - direct download by exact filename**

In [ ]:
response = benchFIMquery(
    file_name='HWM_10_0m_20160928_20161009_780051W352232N_BM.tif',  # exact catalog name
    download=True,            # download it
    out_dir='../downloads/',  # destination folder
)
print(response)

### **Filter combinations**

| Filters | Result |
| --- | --- |
| `file_name` | Direct lookup / download by filename |
| `raster_path` or `boundary_path` (AOI) | Spatial search; `area=True` adds overlap |
| AOI + `event_date` | Match on an exact event date |
| AOI + `start_date` / `end_date` | Match within a date range |
| `start_date` / `end_date` only | All records in the range (no AOI) |
| `huc8`, `tier` | Narrow by basin / benchmark tier |